In [ ]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

# Build Embeddings
This notebook generates embeddings for both SBERT (Sequence encoder) and OpenAI models and stores the indices inside the experiment folder.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [ ]:

import pathlib, json, numpy as np, faiss
from tqdm import tqdm
from src.datasets.dataset import load_data
from src.rag.vector_store import VectorStore
from src.rag import _ARTIFACTS_DIR, _SBERT_DIR, _OPENAI_DIR
from src.embeddings.openai_embedder import OpenAIEmbedder
from sentence_transformers import SentenceTransformer

# ---- Parameters ----
N_CLASSES = int(os.getenv('N_CLASSES', '10'))
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_MODEL = "text-embedding-3-small"

print('Artifacts root:', _ARTIFACTS_DIR)
_SBERT_DIR.mkdir(parents=True, exist_ok=True)
_OPENAI_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# Load dataset
X_train, y_train, _, _, _ = load_data(n_classes=N_CLASSES)
print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


In [ ]:

# ---- SBERT embeddings ----
sbert = SentenceTransformer(SBERT_MODEL)
vectors = sbert.encode(X_train, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')

meta = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, vectors)):
    meta.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

faiss_path = _SBERT_DIR / "index.faiss"
meta_path  = _SBERT_DIR / "meta.jsonl"
VectorStore.build(vectors, meta, vectors.shape[1], faiss_path, meta_path)
print("✅ SBERT index saved at", faiss_path)


In [ ]:

# ---- OpenAI embeddings ----
# Requires OPENAI_API_KEY env var
openai_embedder = OpenAIEmbedder(model=OPENAI_MODEL, batch_size=50)
openai_vecs = openai_embedder.encode(X_train)
openai_vecs = np.array(openai_vecs, dtype='float32')

meta_openai = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, openai_vecs)):
    meta_openai.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

openai_faiss = _OPENAI_DIR / "index.faiss"
openai_meta  = _OPENAI_DIR / "meta.jsonl"
VectorStore.build(openai_vecs, meta_openai, openai_vecs.shape[1], openai_faiss, openai_meta)
print("✅ OpenAI index saved at", openai_faiss)
